# Mu-SHROOM Hindi: Hindi fine-tuning + official scoring

This notebook is designed for:
1. Fine-tune **Hindi-specific XLM-R** and evaluate on Hindi (`HI -> HI`).
2. Optimize post-processing on a Hindi dev split for stronger official **IoU/Correlation**.
3. Compare against a fixed EN->HI baseline score row from your English notebook.
4. Score with the official Mu-SHROOM `participant_kit/scorer.py`.

In [1]:
# Setup
!pip install -q "datasets>=2.18.0" "transformers>=4.41.0" "accelerate>=0.29.0" evaluate seqeval scikit-learn

import inspect
import importlib.util
import json
import os
import random
import subprocess
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
torch: 2.10.0+cu128
cuda: True
gpu: NVIDIA L4


In [2]:
# Experiment configuration
MODEL_NAME = "xlm-roberta-large"
LANGUAGE = "hi"
SEED = 1234
MAX_LENGTH = 512

OUTPUT_ROOT = "/content/mu-shroom-hi-xlmr-large"
HINDI_CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "best")
PREDICTIONS_DIR = os.path.join(OUTPUT_ROOT, "predictions")
SCORER_REPO_DIR = "/content/mu-shroom"

RUN_HINDI_TRAINING = True
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
NUM_EPOCHS = 20
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 3

# Hardcoded best post-processing params from prior Hindi dev search
PREDICTION_THRESHOLD = 0.60
PREDICTION_TEMPERATURE = 1.20
MIN_SPAN_LEN = 1
MERGE_GAP = 2

# Keep False for clean/faster reruns
RUN_DEV_PARAM_SEARCH = False

label_list = ["O", "B-HALL", "I-HALL"]
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("Language:", LANGUAGE)
print("Output root:", OUTPUT_ROOT)
print("Hardcoded inference params:", PREDICTION_THRESHOLD, PREDICTION_TEMPERATURE, MERGE_GAP, MIN_SPAN_LEN)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Language: hi
Output root: /content/mu-shroom-hi-xlmr-large
Hardcoded inference params: 0.6 1.2 2 1


In [3]:
# Shared helpers
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_mu_shroom_language(language: str) -> DatasetDict:
    try:
        ds = load_dataset("Helsinki-NLP/mu-shroom", language)
        print(f"Loaded direct config: {language}")
        return ds
    except Exception as direct_error:
        print(f"Direct config load failed for {language}: {direct_error}")
        ds = load_dataset("Helsinki-NLP/mu-shroom", "all")
        for candidate_column in ["language", "lang", "locale"]:
            if candidate_column in ds[next(iter(ds.keys()))].column_names:
                return DatasetDict(
                    {
                        split_name: split_ds.filter(lambda row: row[candidate_column].lower() == language)
                        for split_name, split_ds in ds.items()
                    }
                )
        raise RuntimeError("Could not locate language column for filtering.")


def infer_schema(example_row: Dict) -> Dict[str, Optional[str]]:
    columns = list(example_row.keys())
    def choose(cands):
        for c in cands:
            if c in columns:
                return c
        return None
    return {
        "text_col": choose(["model_output_text", "answer", "output_text", "text", "response"]),
        "hard_col": choose(["hard_labels", "hard_spans", "hallucination_spans"]),
        "soft_col": choose(["soft_labels", "soft_spans", "probabilistic_labels"]),
        "id_col": choose(["id", "sample_id", "instance_id"]),
    }


def normalize_hard_spans(raw_spans) -> List[List[int]]:
    if raw_spans is None:
        return []
    normalized = []
    for span in raw_spans:
        if span is None:
            continue
        if isinstance(span, dict):
            start, end = span.get("start"), span.get("end")
        elif isinstance(span, (list, tuple)) and len(span) >= 2:
            start, end = span[0], span[1]
        else:
            continue
        if start is None or end is None:
            continue
        start, end = int(start), int(end)
        if end > start:
            normalized.append([start, end])
    return sorted(normalized, key=lambda x: (x[0], x[1]))


def token_overlaps_span(token_start: int, token_end: int, span_start: int, span_end: int) -> bool:
    return (token_start < span_end) and (token_end > span_start)


def spans_to_token_bio(offset_mapping, raw_spans, label2id_map) -> List[int]:
    spans = normalize_hard_spans(raw_spans)
    labels, active_span_index = [], None
    for token_start, token_end in offset_mapping:
        token_start, token_end = int(token_start), int(token_end)
        if token_start == token_end == 0:
            labels.append(-100)
            continue
        matched_index = None
        for span_index, (span_start, span_end) in enumerate(spans):
            if token_overlaps_span(token_start, token_end, span_start, span_end):
                matched_index = span_index
                break
        if matched_index is None:
            labels.append(label2id_map["O"])
            active_span_index = None
            continue
        span_start, _ = spans[matched_index]
        begins_here = token_start <= span_start < token_end or token_start == span_start or active_span_index != matched_index
        labels.append(label2id_map["B-HALL"] if begins_here else label2id_map["I-HALL"])
        active_span_index = matched_index

    prev_inside = False
    for i, lab in enumerate(labels):
        if lab in (-100, label2id_map["O"]):
            prev_inside = False
            continue
        if lab == label2id_map["I-HALL"] and not prev_inside:
            labels[i] = label2id_map["B-HALL"]
        prev_inside = True
    return labels


def merge_spans(spans: List[List[int]], gap: int = 0) -> List[List[int]]:
    spans = normalize_hard_spans(spans)
    if not spans:
        return []
    merged = [spans[0][:]]
    for start, end in spans[1:]:
        ps, pe = merged[-1]
        if start <= pe + gap:
            merged[-1][1] = max(pe, end)
        else:
            merged.append([start, end])
    return merged


def token_probs_to_soft_labels(offset_mapping, hall_probs: List[float]) -> List[Dict[str, float]]:
    out = []
    for (token_start, token_end), prob in zip(offset_mapping, hall_probs):
        token_start, token_end = int(token_start), int(token_end)
        if token_start >= token_end:
            continue
        out.append({"start": token_start, "end": token_end, "prob": float(prob)})
    return out


def preprocess_batch(batch, tokenizer_obj, text_column: str, hard_column: str, max_length: int = 512):
    encodings = tokenizer_obj(batch[text_column], truncation=True, max_length=max_length, return_offsets_mapping=True)
    encodings["labels"] = [
        spans_to_token_bio(offsets, raw_spans, label2id)
        for offsets, raw_spans in zip(encodings["offset_mapping"], batch[hard_column])
    ]
    return encodings


def flatten_non_ignored(labels: List[List[int]]) -> np.ndarray:
    return np.array([int(v) for row in labels for v in row if int(v) != -100], dtype=np.int64)


def compute_class_weights(train_dataset, num_labels: int) -> torch.Tensor:
    y = flatten_non_ignored(train_dataset["labels"])
    classes = np.arange(num_labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return torch.tensor(weights, dtype=torch.float32)


def compute_eval_metrics(eval_pred):
    logits, labels = eval_pred
    pred_ids = np.argmax(logits, axis=-1)
    true_flat, pred_flat = [], []
    for pred_row, gold_row in zip(pred_ids, labels):
        for pred_id, gold_id in zip(pred_row, gold_row):
            if int(gold_id) == -100:
                continue
            true_flat.append(int(gold_id))
            pred_flat.append(int(pred_id))
    true_flat = np.array(true_flat)
    pred_flat = np.array(pred_flat)

    f1_macro = float(f1_score(true_flat, pred_flat, average="macro", zero_division=0))
    f1_b = float(f1_score(true_flat, pred_flat, labels=[label2id["B-HALL"]], average="macro", zero_division=0))
    f1_i = float(f1_score(true_flat, pred_flat, labels=[label2id["I-HALL"]], average="macro", zero_division=0))
    f1_hall_mean = (f1_b + f1_i) / 2.0
    return {"f1_macro": f1_macro, "f1_B-HALL": f1_b, "f1_I-HALL": f1_i, "f1_hall_mean": f1_hall_mean}


class WeightedTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_weights = class_weights.detach().clone()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self._ce_weights.to(device=logits.device, dtype=logits.dtype), ignore_index=-100)
        loss = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def make_training_args(output_dir: str) -> TrainingArguments:
    sig = inspect.signature(TrainingArguments.__init__)
    params = sig.parameters
    kwargs = dict(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=10,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1_hall_mean",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to="none",
    )
    if "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "epoch"
        kwargs["save_strategy"] = "epoch"
    else:
        if "eval_strategy" in params:
            kwargs["eval_strategy"] = "epoch"
        if "save_strategy" in params:
            kwargs["save_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def json_default(obj):
    if hasattr(obj, "item"):
        return obj.item()
    if isinstance(obj, set):
        return list(obj)
    raise TypeError(f"Type not serializable: {type(obj)}")


def write_jsonl(path: str, rows: List[Dict]):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, default=json_default) + "\n")


def clone_if_missing(url: str, path: str):
    if os.path.isdir(path):
        return
    subprocess.check_call(["git", "clone", "--depth", "1", url, path])


def load_scorer_module(repo_dir: str):
    scorer_path = os.path.join(repo_dir, "participant_kit", "scorer.py")
    if not os.path.isfile(scorer_path):
        raise FileNotFoundError(f"Expected scorer.py at {scorer_path}")
    spec = importlib.util.spec_from_file_location("mu_shroom_scorer", scorer_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def build_reference_jsonl(split_ds: Dataset, schema_map: Dict[str, Optional[str]], out_path: str):
    text_col = schema_map["text_col"]
    hard_col = schema_map["hard_col"]
    soft_col = schema_map["soft_col"]
    id_col = schema_map["id_col"] or "id"
    rows = []
    for row_index, row in enumerate(split_ds):
        rows.append({
            "id": row.get(id_col, row_index),
            "model_output_text": row[text_col],
            "soft_labels": row.get(soft_col, []),
            "hard_labels": normalize_hard_spans(row.get(hard_col, [])),
        })
    write_jsonl(out_path, rows)
    return out_path


def score_predictions_with_official_scorer(reference_jsonl: str, prediction_jsonl: str, score_output_txt: str):
    clone_if_missing("https://github.com/Helsinki-NLP/mu-shroom", SCORER_REPO_DIR)
    scorer = load_scorer_module(SCORER_REPO_DIR)
    reference_records = scorer.load_jsonl_file_to_records(reference_jsonl, is_ref=True)
    prediction_records = scorer.load_jsonl_file_to_records(prediction_jsonl, is_ref=False)
    ious, correlations = scorer.main(reference_records, prediction_records, score_output_txt)
    return float(np.mean(ious)), float(np.mean(correlations))


def predict_dataset_to_official_rows(
    checkpoint_dir: str,
    dataset_split: Dataset,
    schema_map: Dict[str, Optional[str]],
    threshold: float = 0.5,
    temperature: float = 1.0,
    min_span_len: int = 1,
    merge_gap: int = 0,
    tokenizer_name: Optional[str] = None,
):
    checkpoint_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name or checkpoint_dir, use_fast=True)
    checkpoint_model = AutoModelForTokenClassification.from_pretrained(checkpoint_dir)
    checkpoint_model.to(DEVICE)
    checkpoint_model.eval()

    text_col = schema_map["text_col"]
    id_col = schema_map["id_col"] or "id"
    checkpoint_id2label = {int(k): v for k, v in checkpoint_model.config.id2label.items()}
    hall_ids = [idx for idx, name in checkpoint_id2label.items() if name in {"B-HALL", "I-HALL"}]

    rows, token_metric_gold, token_metric_pred = [], [], []
    with torch.no_grad():
        for row_index, row in enumerate(dataset_split):
            text = row[text_col]
            encoded = checkpoint_tokenizer(text, return_offsets_mapping=True, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
            offset_mapping = encoded.pop("offset_mapping")[0].tolist()
            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

            logits = checkpoint_model(**encoded).logits[0]
            probabilities = F.softmax(logits / max(float(temperature), 1e-6), dim=-1)
            pred_ids = logits.argmax(dim=-1).tolist()

            hall_probs = []
            for token_index in range(probabilities.shape[0]):
                hall_prob = float(probabilities[token_index, hall_ids].sum().item()) if hall_ids else 0.0
                hall_probs.append(hall_prob)

            thresholded_spans = []
            for (start, end), prob, pred_id in zip(offset_mapping, hall_probs, pred_ids):
                start, end = int(start), int(end)
                if start >= end:
                    continue
                if prob >= threshold and checkpoint_id2label[int(pred_id)] in {"B-HALL", "I-HALL"}:
                    thresholded_spans.append([start, end])
            thresholded_spans = [sp for sp in thresholded_spans if (sp[1] - sp[0]) >= int(min_span_len)]
            hard_spans = merge_spans(thresholded_spans, gap=merge_gap)

            rows.append({
                "id": row.get(id_col, row_index),
                "hard_labels": hard_spans,
                "soft_labels": token_probs_to_soft_labels(offset_mapping, hall_probs),
            })

            if schema_map["hard_col"] is not None:
                gold_labels = spans_to_token_bio(offset_mapping, row[schema_map["hard_col"]], label2id)
                for gold_label, pred_id, (start, end) in zip(gold_labels, pred_ids, offset_mapping):
                    if int(start) == int(end) == 0 or gold_label == -100:
                        continue
                    token_metric_gold.append(gold_label)
                    pred_name = checkpoint_id2label[int(pred_id)]
                    token_metric_pred.append(label2id.get(pred_name, label2id["O"]))

    token_metrics = {}
    if token_metric_gold:
        precision, recall, f1_hall, _ = precision_recall_fscore_support(
            token_metric_gold, token_metric_pred,
            labels=[label2id["B-HALL"], label2id["I-HALL"]], average="micro", zero_division=0,
        )
        token_metrics = {
            "token_precision_hall": float(precision),
            "token_recall_hall": float(recall),
            "token_f1_hall_micro": float(f1_hall),
            "token_f1_macro": float(f1_score(token_metric_gold, token_metric_pred, average="macro", zero_division=0)),
        }
    return rows, token_metrics

In [4]:
# Load Hindi dataset and prepare disjoint train/dev/final eval splits
raw_ds = load_mu_shroom_language(LANGUAGE)
print(raw_ds)
print("Available splits:", list(raw_ds.keys()))
for split_name, split_ds in raw_ds.items():
    print(f"{split_name}: {len(split_ds)} rows")

preview_split_name = "train" if "train" in raw_ds else next(iter(raw_ds.keys()))
schema = infer_schema(raw_ds[preview_split_name][0])
print("Schema:", schema)
if schema["text_col"] is None or schema["hard_col"] is None:
    raise ValueError("Could not infer text/hard label columns.")

text_col = schema["text_col"]
hard_col = schema["hard_col"]
id_col = schema["id_col"] or "id"

if "train" in raw_ds and "validation" in raw_ds:
    # Preferred setup: train/dev from official train, final evaluation on official validation.
    split_train = raw_ds["train"].train_test_split(test_size=0.15, seed=SEED)
    train_raw, dev_raw = split_train["train"], split_train["test"]
    final_eval_raw = raw_ds["validation"]
else:
    # Fallback setup: no official train split. Build disjoint subsets from validation.
    base_split_name = "validation" if "validation" in raw_ds else next(iter(raw_ds.keys()))
    base_ds = raw_ds[base_split_name]

    # 80% train, 20% holdout
    split_1 = base_ds.train_test_split(test_size=0.20, seed=SEED)
    train_raw = split_1["train"]

    # Split holdout equally: 10% dev, 10% final eval (both disjoint from train)
    split_2 = split_1["test"].train_test_split(test_size=0.50, seed=SEED)
    dev_raw = split_2["train"]
    final_eval_raw = split_2["test"]

train_ids = set(train_raw[id_col]) if id_col in train_raw.column_names else set(range(len(train_raw)))
dev_ids = set(dev_raw[id_col]) if id_col in dev_raw.column_names else set(range(len(dev_raw)))
eval_ids = set(final_eval_raw[id_col]) if id_col in final_eval_raw.column_names else set(range(len(final_eval_raw)))
print("train_raw:", len(train_raw))
print("dev_raw:", len(dev_raw))
print("final_eval_raw:", len(final_eval_raw))
print("overlap(train, dev):", len(train_ids & dev_ids))
print("overlap(train, eval):", len(train_ids & eval_ids))
print("overlap(dev, eval):", len(dev_ids & eval_ids))

train_tok = train_raw.map(
    lambda batch: preprocess_batch(batch, tokenizer, text_column=text_col, hard_column=hard_col),
    batched=True,
    remove_columns=train_raw.column_names,
)
dev_tok = dev_raw.map(
    lambda batch: preprocess_batch(batch, tokenizer, text_column=text_col, hard_column=hard_col),
    batched=True,
    remove_columns=dev_raw.column_names,
)

class_weights = compute_class_weights(train_tok, len(label_list))
print("Class weights [O, B-HALL, I-HALL]:", class_weights.tolist())

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)



README.md: 0.00B [00:00, ?B/s]

hi/validation-00000-of-00001.parquet:   0%|          | 0.00/52.9k [00:00<?, ?B/s]

hi/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]

Loaded direct config: hi
DatasetDict({
    validation: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'soft_labels', 'hard_labels', 'model_output_logits', 'model_output_tokens', 'wikipedia_url', 'annotations'],
        num_rows: 50
    })
    test: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'soft_labels', 'hard_labels', 'model_output_logits', 'model_output_tokens', 'wikipedia_url', 'annotations'],
        num_rows: 150
    })
})
Available splits: ['validation', 'test']
validation: 50 rows
test: 150 rows
Schema: {'text_col': 'model_output_text', 'hard_col': 'hard_labels', 'soft_col': 'soft_labels', 'id_col': 'id'}
train_raw: 40
dev_raw: 5
final_eval_raw: 5
overlap(train, dev): 0
overlap(train, eval): 0
overlap(dev, eval): 0


Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Class weights [O, B-HALL, I-HALL]: [0.7600472569465637, 8.808218955993652, 0.6366336345672607]


In [5]:
# Fine-tune Hindi checkpoint
if RUN_HINDI_TRAINING:
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_list),
        label2id=label2id,
        id2label=id2label,
    )

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=make_training_args(OUTPUT_ROOT),
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=data_collator,
        compute_metrics=compute_eval_metrics,
    )
    trainer.train()

    os.makedirs(HINDI_CHECKPOINT_DIR, exist_ok=True)
    trainer.model.save_pretrained(HINDI_CHECKPOINT_DIR, safe_serialization=True)
    tokenizer.save_pretrained(HINDI_CHECKPOINT_DIR)
    print("Saved Hindi checkpoint to:", HINDI_CHECKPOINT_DIR)
else:
    print("RUN_HINDI_TRAINING=False; expecting checkpoint at:", HINDI_CHECKPOINT_DIR)

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 B-hall,F1 I-hall,F1 Hall Mean
1,No log,1.091532,0.255924,0.000000,0.000000,0.000000
2,8.314302,0.931108,0.354164,0.311111,0.000000,0.155556
3,8.314302,0.838370,0.361176,0.305085,0.000000,0.152542
4,6.958320,0.821056,0.427461,0.451613,0.000000,0.225806
5,6.958320,0.830410,0.440594,0.500000,0.000000,0.250000
6,5.594748,0.812495,0.455816,0.428571,0.108108,0.268340
7,5.594748,0.770195,0.521637,0.533333,0.200000,0.366667
8,4.031326,0.893963,0.553491,0.545455,0.292683,0.419069
9,4.031326,0.836592,0.621084,0.560000,0.458333,0.509167
10,3.113693,0.801296,0.715543,0.592593,0.686567,0.639580


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Hindi checkpoint to: /content/mu-shroom-hi-xlmr-large/best


In [6]:
# Build final Hindi reference file for official scorer
reference_jsonl = os.path.join(PREDICTIONS_DIR, "reference_hi_eval.jsonl")
build_reference_jsonl(final_eval_raw, schema, reference_jsonl)
print("Reference file:", reference_jsonl)

Reference file: /content/mu-shroom-hi-xlmr-large/predictions/reference_hi_eval.jsonl


In [7]:
# EN -> HI baseline scores (from EnglishFinetuning.ipynb)
en_to_hi_results = {
    "Model": "English-finetuned XLM-R on Hindi",
    "IoU": 0.42169396198089193,
    "Correlation": 0.398120,
    "Prediction file": "",
}

display(pd.DataFrame([en_to_hi_results]))

,Model,IoU,Correlation,Prediction file
0,English-finetuned XLM-R on Hindi,0.421694,0.39812,


In [8]:
# Evaluate Hindi-finetuned checkpoint on Hindi (HI -> HI)
hi_to_hi_prediction_jsonl = os.path.join(PREDICTIONS_DIR, "hi_to_hi_predictions.jsonl")
hi_to_hi_scores_txt = os.path.join(PREDICTIONS_DIR, "hi_to_hi_scores.txt")
hi_to_hi_results = None

if not os.path.isdir(HINDI_CHECKPOINT_DIR):
    print("Hindi checkpoint not found:", HINDI_CHECKPOINT_DIR)
    print("Run training cell first or set HINDI_CHECKPOINT_DIR.")
else:
    hi_rows, _ = predict_dataset_to_official_rows(
        checkpoint_dir=HINDI_CHECKPOINT_DIR,
        dataset_split=final_eval_raw,
        schema_map=schema,
        threshold=PREDICTION_THRESHOLD,
        temperature=PREDICTION_TEMPERATURE,
        min_span_len=MIN_SPAN_LEN,
        merge_gap=MERGE_GAP,
    )
    write_jsonl(hi_to_hi_prediction_jsonl, hi_rows)
    hi_iou, hi_corr = score_predictions_with_official_scorer(reference_jsonl, hi_to_hi_prediction_jsonl, hi_to_hi_scores_txt)
    hi_to_hi_results = {
        "Model": "Hindi-finetuned XLM-R on Hindi",
        "IoU": hi_iou,
        "Correlation": hi_corr,
        "Prediction file": hi_to_hi_prediction_jsonl,
    }
    print(
        f"Using hardcoded params: thr={PREDICTION_THRESHOLD}, temp={PREDICTION_TEMPERATURE}, "
        f"gap={MERGE_GAP}, min_len={MIN_SPAN_LEN}"
    )
    display(pd.DataFrame([hi_to_hi_results]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Using hardcoded params: thr=0.6, temp=1.2, gap=2, min_len=1


,Model,IoU,Correlation,Prediction file
0,Hindi-finetuned XLM-R on Hindi,0.851852,0.483811,/content/mu-shroom-hi-xlmr-large/predictions/h...


In [9]:
# Final comparison table
comparison_rows = []
if en_to_hi_results is not None:
    comparison_rows.append(en_to_hi_results)
if hi_to_hi_results is not None:
    comparison_rows.append(hi_to_hi_results)

if comparison_rows:
    comparison_df = (
        pd.DataFrame(comparison_rows)[["Model", "IoU", "Correlation", "Prediction file"]]
        .sort_values(by="IoU", ascending=False)
        .reset_index(drop=True)
    )
    display(comparison_df)
else:
    print("No results yet. Run Hindi evaluation cells first.")

,Model,IoU,Correlation,Prediction file
0,Hindi-finetuned XLM-R on Hindi,0.851852,0.483811,/content/mu-shroom-hi-xlmr-large/predictions/h...
1,English-finetuned XLM-R on Hindi,0.421694,0.398120,
